<a href="https://colab.research.google.com/github/yogeshsahu04/gemini-gen-ai-poc/blob/main/Gemini_api_candidate_count_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import google.generativeai as genai
import os
from google.colab import userdata
import json

# Configure the Gemini API key
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except AttributeError:
    API_KEY = os.environ.get('GEMINI_API_KEY')

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Please set it in Colab secrets or environment variables.")

genai.configure(api_key=API_KEY)

# Initialize the Generative Model with a candidate count
# It's important to set a candidate_count greater than 1 to test this feature
model = genai.GenerativeModel(
    'gemini-2.5-flash', # Changed model name to gemini-2.5-flash based on available models
    generation_config={
        'candidate_count': 3, # As requested by the user for testing
        'max_output_tokens': 4096, # Increased maximum number of tokens to allow for complete JSON output
        'temperature': 0.8,       # Controls the randomness of the output (increased for diversity)
        'top_p': 1.0               # Nucleus sampling parameter
    },
    safety_settings=[
        {
            "category": "HARM_CATEGORY_HARASSMENT",
            "threshold": "BLOCK_NONE"
        },
        {
            "category": "HARM_CATEGORY_HATE_SPEECH",
            "threshold": "BLOCK_NONE"
        },
        {
            "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
            "threshold": "BLOCK_NONE"
        },
        {
            "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
            "threshold": "BLOCK_NONE"
        },
    ]
)

# User defined prompt and message text. These can be changed to test different scenarios.
prompt_input = "# Role Definition\nrole: Communication Surveillance Analyst\nobjective: Thoroughly evaluate if a given communication (chat, email, or message) triggers any surveillance rules related to banking/financial compliance.\n\n# Input\n- Text: contains message text body\n\n# Output Schema\n{\n  \"MsgSumm\": \"Short 1–2 line summary of the message text\",\n  \"ruleDetections\": [\n    {\n      \"ruleId\": \"string\",              # ID of evaluation rule (e.g., ER120)\n      \"ruleName\": \"string\",            # Name of evaluation rule (e.g., Market Manipulation)\n      \"explanation\": \"string\",         # Brief reason why the rule was detected\n      \"cite\": \"string\",                # Exact line(s) or sentence(s) from the text where rule was detected, comma-separated if multiple\n      \"ruleDetected\": true             # Boolean flag, always true if rule triggered\n    }\n  ],\n  \"modelThoughts\": {\n    \"summary\": \"Brief description of reasoning process\",\n    \"tokens\": {\n      \"inputTokens\": \"integer\",        # Number of tokens in input\n      \"outputTokens\": \"integer\",       # Number of tokens in output\n      \"cachedTokens\": \"integer\",       # Number of tokens retrieved from cache\n      \"thinkingTokens\": \"integer\"      # Number of tokens used for reasoning/thinking\n    }\n  }\n}\n\n# Notes\n- If **no rules are detected**, only populate `MsgSumm` and `modelThoughts`.\n- Accuracy is the main focus: rules should only be flagged if clearly present.\n- Candidate Count: To request multiple distinct responses, add → \"candidate_count: 3\" (where N is the number of variations required).\n- **Output Format:** The output must be raw JSON, do not wrap it in markdown code fences.\n- The prompt, rules, and message inputs are provided in YAML format.\n- The message input may contain HTML tags, which should be ignored for content analysis.\n"
evaluation_rule_input = "\nevaluation rules:\n- rule_id: ER120\n  rule_name: Market Manipulation\n  description: Detects conversations suggesting attempts to distort, misrepresent, or artificially influence market prices, client trades, or financial instruments.\n\n- rule_id: ER121\n  rule_name: Abusive Language\n  description: Flags use of offensive, discriminatory, or threatening language directed at clients, colleagues, or institutions.\n\n- rule_id: ER122\n  rule_name: Misrepresentation of Numbers\n  description: Identifies attempts to falsify, exaggerate, or conceal financial figures, trade volumes, or performance metrics.\n\n- rule_id: ER123\n  rule_name: Insider Information Disclosure\n  description: Detects sharing of non-public, material information about securities, clients, or corporate actions that could lead to unfair trading advantage.\n\n- rule_id: ER124\n  rule_name: Client Manipulation\n  description: Flags communications that pressure, mislead, or coerce clients into trades or financial decisions against their best interest.\n\n- rule_id: ER125\n  rule_name: Collusion\n  description: Identifies discussions suggesting coordination with other parties to fix prices, rig bids, or manipulate markets.\n\n- rule_id: ER126\n  rule_name: Unauthorized Trade Discussion\n  description: Detects conversations about executing trades or transactions outside approved channels, policies, or without proper authorization.\n\n- rule_id: ER127\n  rule_name: Fraudulent Intent\n  description: Flags language indicating intent to deceive, commit fraud, or conceal material facts in financial dealings.\n\n- rule_id: ER128\n  rule_name: Regulatory Evasion\n  description: Identifies attempts to bypass, ignore, or conceal activities from regulators, auditors, or compliance monitoring.\n\n- rule_id: ER129\n  rule_name: Conflict of Interest\n  description: Detects communications suggesting personal gain at the expense of client interests or institutional integrity."
message_text_input = "text: { <html><b>Internal Memo:</b> Our Q4 earnings will be much higher than expected, but don't tell the public until next month. Buy more stock now!</html>\n<html><i>Client Call:</i> You really need to invest heavily in Company X; their stock is about to skyrocket based on some confidential info I just got. Trust me on this one.</html>}"

# Combine prompt_input and message_text_input for the generate_content call
# Ensure all parts of the prompt are included as per the user's request.
full_prompt = f"{prompt_input}\n{evaluation_rule_input}\n{message_text_input}"

# Get token count for the prompt
prompt_token_count = 0
try:
    prompt_token_count = model.count_tokens(full_prompt).total_tokens
except Exception as e:
    print(f"Could not count prompt tokens: {e}")


# Make a generate_content call
try:
    response = model.generate_content(full_prompt)

    output_data = {
        "tokenDetails": {
            "promptTokens": prompt_token_count,
            "responseTokens": 0 # Will update after processing candidates
        },
        "candidates": []
    }

    total_response_tokens = 0

    if response.candidates:
        for i, candidate in enumerate(response.candidates):
            candidate_output = {
                "index": i, # Added for segregation
                "msgSummary": None,
                "ruleDetections": [],
                "modelThoughts": {
                    "summary": "Could not parse model thoughts from candidate text.",
                    "tokens": {
                        "inputTokens": None,
                        "outputTokens": None,
                        "cachedTokens": None,
                        "thinkingTokens": None
                    }
                }
            }

            # Attempt to parse candidate.text as JSON according to the schema
            try:
                # Access content through candidate.content.parts[0].text
                generated_text = candidate.content.parts[0].text
                # Remove markdown code block fences if present
                if generated_text.startswith('```json') and generated_text.endswith('```'):
                    generated_text = generated_text[len('```json'):-len('```')].strip()
                parsed_candidate_text = json.loads(generated_text)
                candidate_output["msgSummary"] = parsed_candidate_text.get("MsgSumm")
                candidate_output["ruleDetections"] = parsed_candidate_text.get("ruleDetections", [])
                candidate_output["modelThoughts"] = parsed_candidate_text.get("modelThoughts", candidate_output["modelThoughts"])

            except json.JSONDecodeError as e:
                candidate_output["msgSummary"] = f"Error parsing JSON from model output: {e}. Raw text: {generated_text if 'generated_text' in locals() else 'N/A'}"
                # Fallback: if JSON parsing fails, still try to get safety ratings and citation metadata
                if candidate.safety_ratings:
                    for rating in candidate.safety_ratings:
                        rule_detection = {
                            "ruleId": rating.category.value,
                            "ruleName": rating.category.name.replace('HARM_CATEGORY_', '').replace('_', ' ').title(),
                            "explanation": f"Safety Rating Probability: {rating.probability.name}",
                            "cite": None,
                            "ruleDetected": True
                        }
                        candidate_output["ruleDetections"].append(rule_detection)
                if candidate.citation_metadata and candidate.citation_metadata.citations:
                    citations_list = []
                    for citation in candidate.citation_metadata.citations:
                        citations_list.append({
                            "startIndex": citation.start_index,
                            "endIndex": citation.end_index,
                            "uri": citation.uri,
                            "license": citation.license
                        })
                    citation_rule_detection = {
                        "ruleId": "CITATION",
                        "ruleName": "Source Citations",
                        "explanation": "Content generated with verifiable sources.",
                        "cite": json.dumps(citations_list), # Store as JSON string if multiple
                        "ruleDetected": True
                    }
                    candidate_output["ruleDetections"].append(citation_rule_detection)

            output_data["candidates"].append(candidate_output)

            # Count tokens for each candidate's response
            try:
                # Access content through candidate.content.parts[0].text
                candidate_token_count = model.count_tokens(candidate.content.parts[0].text).total_tokens
                total_response_tokens += candidate_token_count
            except Exception as e:
                print(f"Could not count tokens for candidate {i+1}: {e}")

    else:
        # If no candidates, still try to get prompt feedback if available
        if response.prompt_feedback and response.prompt_feedback.block_reason:
            output_data["promptFeedback"] = {
                "blockReason": response.prompt_feedback.block_reason.name,
                "blockReasonCategory": response.prompt_feedback.block_reason_category.name if response.prompt_feedback.block_reason_category else None
            }

    output_data["tokenDetails"]["responseTokens"] = total_response_tokens

    # Print the final JSON output
    print(json.dumps(output_data, indent=2))

except Exception as e:
    print(f"An error occurred during content generation: {e}")
    if "API key not valid" in str(e) or "Authentication failed" in str(e):
        print("Please ensure your GEMINI_API_KEY is correctly set in Colab secrets and is valid.")
    else:
        print("Please check your network connection or try again later.")

{
  "tokenDetails": {
    "promptTokens": 951,
    "responseTokens": 2769
  },
  "candidates": [
    {
      "index": 0,
      "msgSummary": "The communications involve leveraging undisclosed Q4 earnings and confidential market information to urge immediate stock purchases and client investments, while explicitly instructing to delay public disclosure.",
      "ruleDetections": [
        {
          "ruleId": "ER120",
          "ruleName": "Market Manipulation",
          "explanation": "The internal memo encourages buying stock based on undisclosed positive Q4 earnings before public release. The client call advises heavy investment in Company X based on confidential information, both actions attempt to artificially influence market prices or client trades.",
          "cite": "Buy more stock now!, You really need to invest heavily in Company X; their stock is about to skyrocket based on some confidential info I just got.",
          "ruleDetected": true
        },
        {
          